In [1]:
import pandas as pd

cr = pd.read_csv("Outscraper-20260607153436s0a.csv")
imdb = pd.read_csv("IMDB Dataset SPANISH.csv")

cr.head()
imdb.head()

,Unnamed: 0,review_en,review_es,sentiment,sentimiento
0,0,One of the other reviewers has mentioned that ...,Uno de los otros críticos ha mencionado que de...,positive,positivo
1,1,A wonderful little production. The filming tec...,Una pequeña pequeña producción.La técnica de f...,positive,positivo
2,2,I thought this was a wonderful way to spend ti...,Pensé que esta era una manera maravillosa de p...,positive,positivo
3,3,Basically there's a family where a little boy ...,"Básicamente, hay una familia donde un niño peq...",negative,negativo
4,4,"Petter Mattei's ""Love in the Time of Money"" is...","El ""amor en el tiempo"" de Petter Mattei es una...",positive,positivo


In [2]:
#seleccionamos unicamente las columnas que queremos en el corpus
cr = cr[[
    "review_text",
    "review_rating",
    "review_datetime_utc"
]].copy()
# renombramos las columnas 
cr = cr.rename(columns={
    "review_text": "texto",
    "review_rating": "calificacion",
    "review_datetime_utc": "fecha"
})
#eliminamos los nulos 
cr = cr.dropna(subset=["texto"])


cr.head()

,texto,calificacion,fecha
1,An absolute money grab. Definitely not worth i...,1,06/06/2026 19:24:17
2,Love the hermit crabs!,5,06/06/2026 13:46:56
3,"Un lugar espectacular y, sin duda, una visita ...",3,06/06/2026 02:10:30
4,"El parque es espectacular, las 3 son porque es...",3,06/05/2026 21:24:50
5,Touristy and very overrated. Many other great ...,3,06/05/2026 05:46:47


In [ ]:
#aqui clasificamos 
def obtener_polaridad(rating):
    if rating >= 4:
        return "positivo" #de 4 a 5 estrellas decimos que es positivo 
    elif rating <= 2:
        return "negativo" # de 1 a 2 estrellas decimos que es negativo
    else:
        return "neutral" # 3 estrellas es neutral

#aplicamos la funcion a cada valor de la columna calificacion y lo guardamos en una nueva columna.
cr["polaridad"] = cr["calificacion"].apply(obtener_polaridad)
#agregamos columnas de tipo de lugar 'parque' y la fuente 'googlemaps'
cr["tipo_lugar"] = "parque"
cr["fuente"] = "google_maps"

cr.head()

,texto,calificacion,fecha,polaridad,tipo_lugar,fuente
1,An absolute money grab. Definitely not worth i...,1,06/06/2026 19:24:17,negativo,parque,google_maps
2,Love the hermit crabs!,5,06/06/2026 13:46:56,positivo,parque,google_maps
3,"Un lugar espectacular y, sin duda, una visita ...",3,06/06/2026 02:10:30,neutral,parque,google_maps
4,"El parque es espectacular, las 3 son porque es...",3,06/05/2026 21:24:50,neutral,parque,google_maps
5,Touristy and very overrated. Many other great ...,3,06/05/2026 05:46:47,neutral,parque,google_maps


In [ ]:
#seleccionamos las columnas que vamos a usar 
imdb = imdb[[
    "review_es",
    "sentimiento"
]].copy()
#hacemos una limpieza de valores nulos 
imdb = imdb.dropna(subset=["review_es"])

#aqui pongo la misma estructura para que sea la misma que el corpus de cr 
imdb = imdb.rename(columns={
    "review_es": "texto",
    "sentimiento": "polaridad"
})
#limitamos los datos a 1000 
imdb = imdb.head(1000)
# aqui no tenia la calificacion entonces hice una nueva columna y negativo es 1 y positivo 5 
imdb["calificacion"] = imdb["polaridad"].map({
    "positivo": 5,
    "negativo": 1
})
# agregamos tipo de lugar y fuente para cumplir la estructura que necesitamos 
imdb["tipo_lugar"] = "dataset_respaldo"
imdb["fuente"] = "kaggle_imdb"
imdb["fecha"] = None

imdb.head()

,texto,polaridad,calificacion,tipo_lugar,fuente,fecha
0,Uno de los otros críticos ha mencionado que de...,positivo,5,dataset_respaldo,kaggle_imdb,None
1,Una pequeña pequeña producción.La técnica de f...,positivo,5,dataset_respaldo,kaggle_imdb,None
2,Pensé que esta era una manera maravillosa de p...,positivo,5,dataset_respaldo,kaggle_imdb,None
3,"Básicamente, hay una familia donde un niño peq...",negativo,1,dataset_respaldo,kaggle_imdb,None
4,"El ""amor en el tiempo"" de Petter Mattei es una...",positivo,5,dataset_respaldo,kaggle_imdb,None


In [ ]:
#dwfino el orden exacto de el corpus
columnas = [
    "texto",
    "calificacion",
    "polaridad",
    "tipo_lugar",
    "fuente",
    "fecha"
]
#y aqui filtramos por columnas para que tenga el mismo orden 
cr = cr[columnas]
imdb = imdb[columnas]
#ponemos y unimos los dos corpus y evitamos indices duplicado
corpus_final = pd.concat(
    [cr, imdb],
    ignore_index=True
)

print(corpus_final.shape)

(1376, 6)


In [26]:
corpus_final.to_csv(
    "corpus_unificado_resenas.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV generado correctamente")

CSV generado correctamente
